# RAG: Baseline vs Re-Ranking — IFAB *Laws of the Game 2024/25*

Two retrieval pipelines over the **same chunks** of one long PDF:

1. **Baseline** — Bi-Encoder, top 10 by cosine similarity, top 3 go to the LLM.
2. **Re-ranking** — the same top 10, re-scored by a Cross-Encoder, top 3 go to the LLM.

The notebook is split into stages. Only what is slow or hand-made is saved to disk; the cheap steps are
recomputed on every run, so "Restart and run all" always works.

| Stage | Saved to disk |
|---|---|
| Stage 0 — setup | nothing (reads `data/config.json`) |
| Stage 1 — corpus | nothing (rebuilt from the PDF each run) |
| Stage 2 — chunks | `data/chunks.jsonl`, frozen so chunk ids stay stable |
| Stage 3 — embeddings and index | nothing (recomputed, about a second) |
| Stage 4–5 — retrieval and re-ranking | nothing (recomputed) |
| Stage 6 — queries and rank shifts | `data/queries.json` (hand-made), `results/rank_shifts.csv` |
| Stage 7 — Answer 1 and Answer 2 | `results/answers.json` (so a re-run makes no API calls) |
| Stage 8 — latency | `results/latency.csv` |

## Stage 0 — Setup

Load the settings from `data/config.json` once. Every later stage reads the same values, so the
models, chunk sizes and top-k are never typed in two places.

In [1]:
import json, re, sys, random
from pathlib import Path
from collections import OrderedDict

from pypdf import PdfReader

ROOT = Path.cwd()                                   # this notebook sits in repo/, so cwd is repo/
CFG = json.loads((ROOT / "data" / "config.json").read_text())
PDF_PATH = ROOT / CFG["source_pdf"]
random.seed(CFG["seed"])

print("python :", sys.executable)
print("PDF    :", PDF_PATH.name, "| found:", PDF_PATH.exists())
print("include:", len(CFG["include_sections"]), "sections")

python : /home/noel/Desktop/Obsidian/MTech_CSE/SEM3/Subjects/LMA/Assignments/RAG-Reranking/repo/.venv/bin/python
PDF    : Laws-of-the-Game-2024_25.pdf | found: True
include: 20 sections


## Stage 1 — Corpus: PDF to clean paragraphs

The PDF is a booklet: most PDF pages are **two printed pages side by side**. The text extractor gives the
left page first, then the right one, so reading order is fine. What needs cleaning:

- a **footer** on every page (`Laws of the Game 2024/25 | Law 11 | Offside`). It is useful: it tells us
  which Law the page belongs to, so we read it for a label and then remove it from the text;
- the **printed page numbers** glued together at the top (e.g. `3938`);
- a vertical **side tab** (`Law`, `11`, `Offside`) that comes out as three stray lines;
- **line breaks** in the middle of sentences. Paragraph starts have a leading space, bullets start
  with `•`, and headings look like `2. Offside offence`, so paragraphs can be rebuilt from those clues.

We keep only the sections listed in `include_sections`. The contents page, the "About the Laws"
introduction, the glossary and the "what changed" summary are left out. The glossary and the change list
repeat rule text, which would give near-identical chunks.

In [2]:
# The footer line on every page, e.g. "Laws of the Game 2024/25   |  Law 1  |  The Field of Play"
FOOTER = re.compile(r"Laws of the Game 2024/25\s*\|\s*(.+?)\s*$")

# Official Law titles, used only to recognise and remove the vertical side tab.
LAW_TITLES = {
    1: "The Field of Play", 2: "The Ball", 3: "The Players", 4: "The Players\u2019 Equipment",
    5: "The Referee", 6: "The Other Match Officials", 7: "The Duration of the Match",
    8: "The Start and Restart of Play", 9: "The Ball in and out of Play",
    10: "Determining the Outcome of a Match", 11: "Offside", 12: "Fouls and Misconduct",
    13: "Free Kicks", 14: "The Penalty Kick", 15: "The Throw-in", 16: "The Goal Kick",
    17: "The Corner Kick",
}

def read_pages(pdf_path):
    """Extract text page by page and tag every page with its section (from the footer)."""
    reader = PdfReader(str(pdf_path))
    pages, current = [], "front matter"          # a page with no footer inherits the last section
    for number, page in enumerate(reader.pages, start=1):
        kept, labels = [], []
        for line in (page.extract_text() or "").split("\n"):
            m = FOOTER.search(line)
            if m:
                labels.append(m.group(1).split("|")[0].strip())   # "Law 1  |  The Field..." -> "Law 1"
                line = FOOTER.sub("", line)
            kept.append(line.rstrip())
        if labels:
            current = labels[-1]
        if kept and re.fullmatch(r"\s*\d{1,6}\s*", kept[0]):    # printed page numbers at the top
            kept = kept[1:]
        pages.append({"pdf_page": number, "section": current, "lines": kept})
    return pages

def drop_side_tab(lines, section):
    """Remove the vertical tab that extracts as 'Law' / '<number>' / '<title words>'."""
    m = re.fullmatch(r"Law (\d+)", section)
    if not m:
        return lines
    title, out, i = LAW_TITLES[int(m.group(1))], [], 0
    while i < len(lines):
        if lines[i].strip() == "Law" and i + 1 < len(lines) and re.fullmatch(r"\s*\d+\s*", lines[i + 1]):
            i += 2
            while i < len(lines) and lines[i].strip() and lines[i].strip() in title:
                i += 1
            continue
        out.append(lines[i])
        i += 1
    return out

In [3]:
BULLET = re.compile(r"^[\s\u2002]*\u2022")            # a line that starts with a bullet point
NUM_HEAD = re.compile(r"^\s*\d{1,2}\.\s+\S")           # "2. Offside offence"

def tidy(text):
    """Collapse spacing and fix '( excluding the halfway line )' -> '(excluding the halfway line)'."""
    text = re.sub(r"\s+", " ", text.replace("\u2002", " ")).strip()
    text = re.sub(r"\(\s+", "(", text)
    return re.sub(r"\s+\)", ")", text)

def build_paragraphs(page_lines):
    """Turn (pdf_page, line) pairs into paragraphs and headings by looking at how each line starts."""
    paras, cur = [], None
    def flush():
        nonlocal cur
        if cur and tidy(cur["text"]):
            paras.append({"kind": cur["kind"], "text": tidy(cur["text"]), "pdf_page": cur["pdf_page"]})
        cur = None
    for page, line in page_lines:
        if not line.strip():
            continue
        plain = line.replace("\u2002", " ")
        is_heading = (bool(NUM_HEAD.match(plain)) and len(plain.split()) <= 8
                      and not plain.rstrip().endswith((".", ":", ";", ",")))
        if is_heading:                                    # a heading stands on its own
            flush()
            paras.append({"kind": "heading", "text": tidy(plain), "pdf_page": page})
        elif plain.startswith(" ") or BULLET.match(line) or NUM_HEAD.match(plain):
            flush()                                       # a new paragraph or bullet starts here
            cur = {"kind": "para", "text": plain, "pdf_page": page}
        elif cur is None:
            cur = {"kind": "para", "text": plain, "pdf_page": page}
        else:
            cur["text"] += " " + plain                    # same paragraph, wrapped onto a new line
    flush()
    return paras

In [4]:
pages = read_pages(PDF_PATH)

# Keep only the sections we want, and glue each section's pages together in order, so a paragraph
# that runs over a page break stays in one piece.
by_section = OrderedDict()
for pg in pages:
    if pg["section"] in CFG["include_sections"]:
        lines = drop_side_tab(pg["lines"], pg["section"])
        by_section.setdefault(pg["section"], []).extend((pg["pdf_page"], l) for l in lines)
corpus = {sec: build_paragraphs(lines) for sec, lines in by_section.items()}

# Two kinds of noise survive the steps above, so remove them and PRINT what was removed:
#  - the big page title at the start of each Law (e.g. "The Players"), which repeats the Law's name;
#  - labels from diagrams (e.g. the signal graphics: "CORNER KICK GOAL KICK OFFSIDE ..."), which get glued
#    onto the END of a real paragraph. We cut the labels off and keep the rule that came before them.
KNOWN_ACRONYMS = {"VAR", "AVAR", "IFAB", "FIFA"}
def shouting(word):
    return bool(re.fullmatch(r"[A-Z]{3,}", word)) and word not in KNOWN_ACRONYMS

def clean_noise(p, sec):
    # Return (paragraph or None, note). Cuts the text at the first two SHOUTING words in a row.
    t, m = p["text"], re.fullmatch(r"Law (\d+)", sec)
    if m and p["kind"] == "para" and len(t.split()) <= 8 and t.strip() in LAW_TITLES[int(m.group(1))]:
        return None, "page title"
    if p["kind"] != "para":
        return p, None
    words = t.split()
    for i in range(len(words) - 1):
        if shouting(words[i].strip(".,;:()")) and shouting(words[i + 1].strip(".,;:()")):
            kept = " ".join(words[:i])
            if len(kept.split()) < 6 or kept.startswith("Refer to graphics"):
                return None, "diagram labels only"
            return {**p, "text": kept}, "labels cut: " + " ".join(words[i:i + 6]) + " ..."
    return p, None

removed = []
for sec in corpus:
    kept = []
    for p in corpus[sec]:
        new_p, note = clean_noise(p, sec)
        if note:
            removed.append((sec, note, p["text"]))
        if new_p:
            kept.append(new_p)
    corpus[sec] = kept
print(f"noise handled in {len(removed)} paragraphs:")
for sec, note, text in removed:
    print(f"  [{sec} | {note}]")
    print("      was:", text[:100])
print()

wc = lambda text: len(text.split())
print(f"{'section':<42}{'pdf pages':>10}{'paragraphs':>12}{'words':>8}")
for sec, paras in corpus.items():
    pdf_pages = sorted({p['pdf_page'] for p in paras})
    print(f"{sec:<42}{str(pdf_pages[0]) + '-' + str(pdf_pages[-1]):>10}{len(paras):>12}{sum(wc(p['text']) for p in paras):>8}")

left_out = OrderedDict()
for pg in pages:
    if pg["section"] not in CFG["include_sections"]:
        left_out[pg["section"]] = left_out.get(pg["section"], 0) + 1
print("\nleft out (section: pdf pages):", dict(left_out))

noise handled in 8 paragraphs:
  [Law 3 | page title]
      was: The Players
  [Law 5 | labels cut: FREE KICK Advantage (2) FREE KICK ...]
      was: Referees and other ‘on-field ’ match officials are prohibited from wearing jewellery or any other el
  [Law 5 | diagram labels only]
      was: Refer to graphics for approved referee signals. Indirect free kick FREE KICK FREE KICK Corner kick C
  [Law 5 | labels cut: CORNER KICK GOAL KICK OFFSIDE PENALTY ...]
      was: • any other decision taken in accordance with the Laws of the Game or in conformity with their dutie
  [Law 6 | labels cut: CORNER KICK GOAL KICK OFFSIDE PENALTY ...]
      was: • communicating information about a VAR-related decision to relevant parties CORNER KICK GOAL KICK O
  [Law 6 | diagram labels only]
      was: Refer to graphics for approved assistant referee signals. CORNER KICK GOAL KICK OFFSIDE PENALTY KICK
  [Law 6 | diagram labels only]
      was: FREE KICKFREE KICK Offside in the middle of the field Offside 

In [5]:
# The corpus is not saved: it is rebuilt from the PDF in about a second. Look at a sample instead.
print("\n--- Law 11 (offside), first lines of the rebuilt text ---")
for p in corpus["Law 11"][:8]:
    print(("[heading] " if p["kind"] == "heading" else "    ") + p["text"][:150])


--- Law 11 (offside), first lines of the rebuilt text ---
[heading] 1. Offside position
    It is not an offence to be in an offside position. A player is in an offside position if:
    • any part of the head, body or feet is in the opponents’ half (excluding the halfway line) and
    • any part of the head, body or feet is nearer to the opponents’ goal line than both the ball and the second-last opponent
    The hands and arms of all players, including the goalkeepers, are not considered. For the purposes of determining offside, the upper boundary of the a
    • second-last opponent or
    • last two opponents
[heading] 2. Offside offence


## Stage 2 — Chunks

A **chunk** is the unit the retriever finds, so its size matters:

- **Too long:** the Bi-Encoder reads at most 256 word-pieces, and anything beyond that is silently cut.
  Long chunks also mix several rules into one vector.
- **Too short:** a clause like "unless the referee allows it" loses the rule it modifies.

The rules used here:

1. Never cross a Law boundary. A chunk never mixes Law 11 with Law 12.
2. Respect the numbered headings (`2. Offside offence`). A chunk stays within one heading.
3. Pack whole paragraphs and bullets up to `max_words`. Split only a paragraph that is longer than that,
   at a sentence boundary.
4. If a paragraph ends with a colon (a lead-in like "A player is in an offside position if:"), keep it
   with the bullets that follow it.
5. Merge a tiny last chunk (under `min_words`) into the one before it.

Each chunk also records its **context** (`Law 11 — Offside | 2. Offside offence`), which the next
stage puts in front of the text when it embeds the chunk.

In [6]:
MAX_W, MIN_W = CFG["chunking"]["max_words"], CFG["chunking"]["min_words"]

def split_long(text, max_words):
    """Split one over-long paragraph into pieces of at most max_words, at sentence ends."""
    pieces, buf = [], []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        words = sentence.split()
        while len(words) > max_words:                       # a single huge sentence: cut by words
            if buf:
                pieces.append(" ".join(buf)); buf = []
            pieces.append(" ".join(words[:max_words])); words = words[max_words:]
        if len(buf) + len(words) > max_words and buf:
            pieces.append(" ".join(buf)); buf = []
        buf += words
    if buf:
        pieces.append(" ".join(buf))
    return pieces

def pack(paras, max_words):
    """Greedy packing of paragraphs into chunks of at most max_words."""
    chunks, buf, n = [], [], 0
    for p in paras:
        pieces = split_long(p["text"], max_words) if wc(p["text"]) > max_words else [p["text"]]
        for piece in pieces:
            w = wc(piece)
            if n + w > max_words and buf:
                carry = None
                if len(buf) > 1 and buf[-1]["text"].rstrip().endswith(":"):
                    carry = buf.pop()                        # keep a lead-in with its bullets
                chunks.append(buf)
                buf, n = ([carry], wc(carry["text"])) if carry else ([], 0)
            buf.append({"text": piece, "pdf_page": p["pdf_page"]})
            n += w
    if buf:
        chunks.append(buf)
    # a tiny last chunk joins the one before it, if the result is not much over the limit
    if len(chunks) > 1 and sum(wc(x["text"]) for x in chunks[-1]) < MIN_W \
            and sum(wc(x["text"]) for c in chunks[-2:] for x in c) <= max_words * 1.3:
        last = chunks.pop()
        chunks[-1] += last
    return chunks

def section_title(sec):
    m = re.fullmatch(r"Law (\d+)", sec)
    return f"{sec} \u2014 {LAW_TITLES[int(m.group(1))]}" if m else sec

chunk_records = []
for sec, paras in corpus.items():
    heading, block = "", []
    blocks = []                                              # [(heading, [paragraphs])]
    for p in paras:
        if p["kind"] == "heading":
            if block: blocks.append((heading, block))
            heading, block = p["text"], []
        else:
            block.append(p)
    if block: blocks.append((heading, block))
    for heading, block in blocks:
        for pieces in pack(block, MAX_W):
            text = "\n".join(x["text"] for x in pieces)
            chunk_records.append({
                "id": f"c{len(chunk_records):04d}",
                "section": sec,
                "context": section_title(sec) + (f" | {heading}" if heading else ""),
                "heading": heading,
                "text": text,
                "pdf_page_start": pieces[0]["pdf_page"],
                "pdf_page_end": pieces[-1]["pdf_page"],
                "n_words": wc(text),
            })
print("chunks:", len(chunk_records))

chunks: 230


In [7]:
# Sanity checks, then save.
assert len({c["id"] for c in chunk_records}) == len(chunk_records)             # ids are unique
assert all(c["n_words"] > 0 for c in chunk_records)                            # no empty chunk
corpus_words = sum(wc(p["text"]) for paras in corpus.values() for p in paras if p["kind"] == "para")
chunk_words = sum(c["n_words"] for c in chunk_records)
assert corpus_words == chunk_words, (corpus_words, chunk_words)                # no text lost or duplicated

with open(ROOT / "data" / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunk_records:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

sizes = sorted(c["n_words"] for c in chunk_records)
print(f"words in corpus = words in chunks = {chunk_words}")
print(f"chunk size (words): min {sizes[0]}, median {sizes[len(sizes)//2]}, max {sizes[-1]}")
print(f"chunks under {MIN_W} words: {sum(s < MIN_W for s in sizes)}")
print("\nchunks per section:")
for sec in corpus:
    print(f"  {sec:<42}{sum(c['section'] == sec for c in chunk_records):>4}")

words in corpus = words in chunks = 26693
chunk size (words): min 13, median 131, max 187
chunks under 40 words: 13

chunks per section:
  Notes and modifications                     20
  Law 1                                       19
  Law 2                                        4
  Law 3                                       16
  Law 4                                       11
  Law 5                                       15
  Law 6                                        7
  Law 7                                        5
  Law 8                                        4
  Law 9                                        2
  Law 10                                       8
  Law 11                                       9
  Law 12                                      28
  Law 13                                       6
  Law 14                                       9
  Law 15                                       4
  Law 16                                       4
  Law 17                      

In [8]:
# Does the embedding model read every chunk in full? Count word-pieces with its own tokenizer.
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("sentence-transformers/" + CFG["bi_encoder"])
limit = 256                                                  # all-MiniLM-L6-v2 truncates after 256
lengths = [len(tok(c["context"] + "\n" + c["text"])["input_ids"]) for c in chunk_records]
print(f"word-pieces per chunk (context + text): median {sorted(lengths)[len(lengths)//2]}, max {max(lengths)}")
print(f"chunks longer than {limit}: {sum(n > limit for n in lengths)} of {len(lengths)}")

word-pieces per chunk (context + text): median 169, max 258
chunks longer than 256: 1 of 230


In [9]:
# Look at a few chunks with your own eyes. This is the most useful check of all.
for c in [x for x in chunk_records if x["section"] == "Law 11"][:2] + \
         [x for x in chunk_records if x["section"] == "Law 12"][5:6]:
    print(f"[{c['id']}] {c['context']}   (pdf p.{c['pdf_page_start']}, {c['n_words']} words)")
    print(c["text"])
    print("-" * 80)

[c0111] Law 11 — Offside | 1. Offside position   (pdf p.50, 116 words)
It is not an offence to be in an offside position. A player is in an offside position if:
• any part of the head, body or feet is in the opponents’ half (excluding the halfway line) and
• any part of the head, body or feet is nearer to the opponents’ goal line than both the ball and the second-last opponent
The hands and arms of all players, including the goalkeepers, are not considered. For the purposes of determining offside, the upper boundary of the arm is in line with the bottom of the armpit. A player is not in an offside position if level with the:
• second-last opponent or
• last two opponents
--------------------------------------------------------------------------------
[c0112] Law 11 — Offside | 2. Offside offence   (pdf p.50, 134 words)
A player in an offside position at the moment the ball is played or touched* by a team-mate is only penalised on becoming involved in active play by:
• interfering with 

## Stage 3 — Embed the chunks and build the index

The **Bi-Encoder** reads each chunk on its own and turns it into one vector (384 numbers). A question is
turned into a vector the same way, and the chunks whose vectors point in the most similar direction are the
retrieved ones. Because the vectors are normalised to length 1, the dot product *is* the cosine similarity,
so a plain inner-product FAISS index does exact cosine search.

Each chunk is embedded as `context + "\n" + text`, so the model also sees which Law it belongs to
(for example `Law 11`). This is cheap to recompute (230 chunks), so nothing is saved: the vectors are
rebuilt every run. Everything runs on the CPU, so the later timings do not depend on a busy GPU.

In [10]:
import numpy as np, faiss, time
from sentence_transformers import SentenceTransformer, CrossEncoder

chunks = [json.loads(line) for line in (ROOT / "data" / "chunks.jsonl").read_text(encoding="utf-8").splitlines()]
chunk_ids = [c["id"] for c in chunks]
by_id = {c["id"]: c for c in chunks}
chunk_texts = [c["context"] + "\n" + c["text"] for c in chunks]

bi_encoder = SentenceTransformer(CFG["bi_encoder"], device="cpu")
t0 = time.perf_counter()
chunk_vecs = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False).astype("float32")
t_embed = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_vecs.shape[1])      # exact inner-product search
index.add(chunk_vecs)
print(f"{len(chunks)} chunks -> vectors {chunk_vecs.shape}, embedded in {t_embed:.1f}s")
print("vector length (should be 1):", float(np.linalg.norm(chunk_vecs[0])))
print("index size:", index.ntotal)

230 chunks -> vectors (230, 384), embedded in 1.8s
vector length (should be 1): 1.0
index size: 230


## Stage 4 — Pipeline 1: Bi-Encoder retrieval (the baseline)

`retrieve(question)` embeds the question and asks the index for the **top 10** chunks
(`top_k_candidates`). The baseline sends the first 3 of them (`top_n_context`) to the LLM. Positions 4 to 10
are exactly the chunks the Cross-Encoder gets a chance to rescue.

In [11]:
K, N = CFG["retrieval"]["top_k_candidates"], CFG["retrieval"]["top_n_context"]

def retrieve(question, k=K):
    """Top-k chunks by cosine similarity: a list of (chunk_id, bi_score), best first."""
    q = bi_encoder.encode([question], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idx = index.search(q, k)
    return [(chunk_ids[i], float(s)) for i, s in zip(idx[0], scores[0])]

def baseline_context(question):
    return [cid for cid, _ in retrieve(question)[:N]]

## Stage 5 — Pipeline 2: Cross-Encoder re-ranking

The **Cross-Encoder** reads the question and one chunk *together* in a single pass and outputs one relevance
score. It is slower than the Bi-Encoder (one pass per pair, 10 pairs per question), but it can compare the
words of the question against the words of the chunk, which the Bi-Encoder cannot. We run it only on the
same 10 candidates, re-sort them by its score and keep the top 3. The score is a raw model output, not a
probability; only the order matters.

In [12]:
cross_encoder = CrossEncoder(CFG["cross_encoder"], device="cpu", max_length=512)

def rerank(question, candidates):
    """Re-sort the candidates (list of (chunk_id, bi_score)) by Cross-Encoder score, best first."""
    pairs = [(question, chunk_texts[chunk_ids.index(cid)]) for cid, _ in candidates]
    ce = cross_encoder.predict(pairs, show_progress_bar=False)
    out = [(cid, float(s)) for (cid, _), s in zip(candidates, ce)]
    return sorted(out, key=lambda x: -x[1])

def reranked_context(question):
    return [cid for cid, _ in rerank(question, retrieve(question))[:N]]

In [13]:
# A look at both rankings for one question. Not a test set: just a check that the two pipelines run.
def show(question):
    cands = retrieve(question)
    re = rerank(question, cands)
    bi_rank = {cid: r for r, (cid, _) in enumerate(cands, 1)}
    print("Q:", question)
    print(f"{'new':>3} {'old':>3}  {'chunk':<7}{'bi':>7}{'cross':>8}  section / heading")
    for new, (cid, s) in enumerate(re, 1):
        c = by_id[cid]
        mark = "  <- promoted into top 3" if new <= N < bi_rank[cid] else ""
        print(f"{new:>3} {bi_rank[cid]:>3}  {cid:<7}{dict(cands)[cid]:>7.3f}{s:>8.2f}  {c['section']} / {c['heading'] or '-'}{mark}")

show("Can a player wear a ring or a bracelet during a match?")

Q: Can a player wear a ring or a bracelet during a match?
new old  chunk       bi   cross  section / heading
  1   1  c0059    0.631    6.35  Law 4 / 1. Safety
  2   3  c0062    0.465   -2.52  Law 4 / 3. Colours
  3   4  c0084    0.465   -3.09  Law 5 / 7. Liability of match officials  <- promoted into top 3
  4   9  c0064    0.414   -3.90  Law 4 / 4. Other equipment
  5   2  c0082    0.497   -4.08  Law 5 / 5. Referee’s equipment
  6   8  c0066    0.416   -4.22  Law 4 / 5. Slogans, statements, images and advertising
  7   7  c0063    0.416   -5.10  Law 4 / 4. Other equipment
  8   5  c0060    0.445   -5.20  Law 4 / 2. Compulsory equipment
  9   6  c0083    0.419   -5.67  Law 5 / 7. Liability of match officials
 10  10  c0135    0.402   -6.93  Law 12 / 3. Disciplinary action


## Stage 6 — Test queries and where the gold chunk lands

`data/queries.json` holds the hand-picked questions. Each has a **gold chunk** (the chunk that contains the
answer) and a short gold answer, both read off the chunk text. Four are queries where the Bi-Encoder
was expected to struggle (a keyword trap, an exception, a negation and a vocabulary gap) and two are
controls where it should do fine.

For each question the table shows the rank of the gold chunk under each pipeline. A rank of 1 to 3 means
the LLM will see it. "Promoted" means rank 4 to 10 for the baseline but 1 to 3 after re-ranking.

In [14]:
import pandas as pd

queries = json.loads((ROOT / "data" / "queries.json").read_text(encoding="utf-8"))

def best_rank(order, gold):
    """Rank (1 = best) of the best-placed gold chunk in a list of chunk ids, or None if absent."""
    ranks = [order.index(g) + 1 for g in gold if g in order]
    return min(ranks) if ranks else None

runs = {}                                                    # query id -> both rankings
rows = []
for q in queries:
    cands = retrieve(q["question"])
    reranked = rerank(q["question"], cands)
    bi_order = [cid for cid, _ in cands]
    ce_order = [cid for cid, _ in reranked]
    runs[q["id"]] = {"bi": cands, "ce": reranked}
    r_bi, r_ce = best_rank(bi_order, q["gold_chunks"]), best_rank(ce_order, q["gold_chunks"])
    rows.append({"id": q["id"], "category": q["category"], "gold": ",".join(q["gold_chunks"]),
                 "bi_rank": r_bi, "ce_rank": r_ce,
                 "in_top3_baseline": bool(r_bi and r_bi <= N), "in_top3_rerank": bool(r_ce and r_ce <= N),
                 "promoted": bool(r_bi and r_ce and r_bi > N and r_ce <= N)})
rank_table = pd.DataFrame(rows)
(ROOT / "results").mkdir(exist_ok=True)
rank_table.to_csv(ROOT / "results" / "rank_shifts.csv", index=False)
rank_table

,id,category,gold,bi_rank,ce_rank,in_top3_baseline,in_top3_rerank,promoted
0,q1,keyword trap,c0099,5,1,False,True,True
1,q2,exception,c0138,9,1,False,True,True
2,q3,negation,c0111,8,1,False,True,True
3,q4,vocabulary gap,c0041,6,1,False,True,True
4,q5,control,c0059,1,1,True,True,False
5,q6,control,c0043,1,1,True,True,False


In [15]:
# The full top-10 for each query: old position -> new position, with the Cross-Encoder score.
for q in queries:
    cands, re = runs[q["id"]]["bi"], runs[q["id"]]["ce"]
    old = {cid: r for r, (cid, _) in enumerate(cands, 1)}
    bi_score = dict(cands)
    print(f"{q['id']} [{q['category']}] {q['question']}   gold: {q['gold_chunks']}")
    print(f"  {'new':>3} {'old':>3}  {'chunk':<7}{'bi':>7}{'cross':>8}  section / heading")
    for new, (cid, s) in enumerate(re, 1):
        c = by_id[cid]
        flag = "  <- GOLD" if cid in q["gold_chunks"] else ""
        print(f"  {new:>3} {old[cid]:>3}  {cid:<7}{bi_score[cid]:>7.3f}{s:>8.2f}  {c['section']} / {c['heading'] or '-'}{flag}")
    print()

q1 [keyword trap] Can a goal be scored directly from a kick-off?   gold: ['c0099']
  new old  chunk       bi   cross  section / heading
    1   5  c0099    0.623    9.60  Law 8 / 1. Kick-off  <- GOLD
    2   1  c0167    0.690    8.49  Law 16 / -
    3   4  c0154    0.630    7.79  Law 14 / -
    4   6  c0171    0.617    7.61  Law 17 / -
    5   2  c0103    0.655    5.67  Law 10 / 1. Goal scored
    6   7  c0159    0.609    2.40  Law 14 / 2. Offences and sanctions
    7   3  c0170    0.648    1.48  Law 16 / 2. Offences and sanctions
    8   9  c0169    0.598    0.02  Law 16 / 2. Offences and sanctions
    9  10  c0119    0.595   -1.10  Law 11 / 4. Offences and sanctions
   10   8  c0105    0.608   -2.92  Law 10 / 3. Penalties (penalty shoot-out)

q2 [exception] Is a goalkeeper sent off for handling the ball inside their own penalty area to stop a goal?   gold: ['c0138']
  new old  chunk       bi   cross  section / heading
    1   9  c0138    0.628    6.09  Law 12 / 3. Disciplinary action

## Stage 7 — Answer 1 and Answer 2 (Gemini)

Both pipelines use the **same LLM, the same settings and the same prompt**; only the three chunks pasted into
the prompt differ. The prompt, verbatim:

```
Answer the question using only the context below, which is taken from the IFAB Laws of the Game 2024/25.
If the context does not contain the answer, reply exactly: The context does not contain the answer.
Answer in at most three sentences.

Context:
[1] <section / heading>
<chunk text>

[2] <section / heading>
<chunk text>

[3] <section / heading>
<chunk text>

Question: <question>
Answer:
```

**Controls on the API.** Calls are made one at a time, at least `min_seconds_between_calls` apart. Only
rate-limit and server errors (429, 5xx) are retried, with growing waits (up to 60 s); any other error stops at once.
Each answer is capped at `max_output_tokens`. The key is read from `.env` and never printed.

Answers are saved to `results/answers.json` after every call, so a crash loses nothing. With `USE_SAVED_ANSWERS = True` an answer whose prompt and model have not
changed is reused, so re-running the notebook makes no API calls. Set it to `False` to ask again.

In [16]:
import os, hashlib
from dotenv import load_dotenv

USE_SAVED_ANSWERS = True
ANSWERS_PATH = ROOT / "results" / "answers.json"
LLM = CFG["llm"]

PROMPT_HEAD = (
    "Answer the question using only the context below, which is taken from the IFAB Laws of the Game 2024/25.\n"
    "If the context does not contain the answer, reply exactly: The context does not contain the answer.\n"
    "Answer in at most three sentences.\n\nContext:\n"
)

def build_prompt(question, chunk_id_list):
    blocks = []
    for i, cid in enumerate(chunk_id_list, 1):
        c = by_id[cid]
        blocks.append(f"[{i}] {c['context']}\n{c['text']}")
    return PROMPT_HEAD + "\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"

saved = json.loads(ANSWERS_PATH.read_text(encoding="utf-8")) if ANSWERS_PATH.exists() else {}

_client = None
_last_call = [0.0]

def ask_llm(prompt):
    """One Gemini call. Returns (answer, seconds spent inside the successful API call)."""
    global _client
    if _client is None:
        from google import genai
        load_dotenv(ROOT / ".env")
        key = os.environ.get(LLM["api_key_env"], "").strip()
        assert key, f"No {LLM['api_key_env']} found. Put it in repo/.env"
        _client = genai.Client(api_key=key)
    from google.genai import types
    for attempt in range(LLM["max_retries"] + 1):
        wait = LLM["min_seconds_between_calls"] - (time.time() - _last_call[0])    # the throttle
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        t0 = time.perf_counter()
        try:
            r = _client.models.generate_content(
                model=LLM["model"], contents=prompt,
                config=types.GenerateContentConfig(temperature=LLM["temperature"],
                                                   max_output_tokens=LLM["max_output_tokens"]))
            elapsed = time.perf_counter() - t0
            reason = str(r.candidates[0].finish_reason) if r.candidates else "NO_CANDIDATE"
            assert r.text and reason.endswith("STOP"), f"answer not complete: {reason}"      # e.g. MAX_TOKENS
            return r.text.strip(), elapsed
        except Exception as e:
            if getattr(e, "code", None) not in (429, 500, 502, 503, 504) or attempt == LLM["max_retries"]:
                raise
            time.sleep(min(2 ** (attempt + 2), 60))                                    # 4, 8, 16, 32, 60 ... s

answers = {}
calls_made = 0
for q in queries:
    for pipe, order in (("baseline", runs[q["id"]]["bi"]), ("rerank", runs[q["id"]]["ce"])):
        ctx = [cid for cid, _ in order[:N]]
        prompt = build_prompt(q["question"], ctx)
        sha = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:12]
        key = f"{q['id']}|{pipe}"
        if USE_SAVED_ANSWERS and key in saved and saved[key]["prompt_sha"] == sha and saved[key].get("model") == LLM["model"]:
            answers[key] = saved[key]
            continue
        text, secs = ask_llm(prompt)
        calls_made += 1
        answers[key] = {"qid": q["id"], "pipeline": pipe, "chunks": ctx, "prompt_sha": sha,
                        "answer": text, "llm_seconds": round(secs, 2), "model": LLM["model"]}
        ANSWERS_PATH.write_text(json.dumps({**saved, **answers}, indent=2, ensure_ascii=False), encoding="utf-8")   # save after every call

ANSWERS_PATH.write_text(json.dumps(answers, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(answers)} answers ready ({calls_made} new API calls, {len(answers) - calls_made} reused from disk)")

12 answers ready (1 new API calls, 11 reused from disk)


In [17]:
# Answer 1 (baseline) next to Answer 2 (re-ranked), with the gold answer to compare against.
for q in queries:
    a1, a2 = answers[f"{q['id']}|baseline"], answers[f"{q['id']}|rerank"]
    print(f"{q['id']} [{q['category']}] {q['question']}")
    print(f"  gold chunk {q['gold_chunks']}  | baseline sent {a1['chunks']}  | rerank sent {a2['chunks']}")
    print(f"  GOLD     : {q['gold_answer']}")
    print(f"  ANSWER 1 : {a1['answer']}")
    print(f"  ANSWER 2 : {a2['answer']}")
    print()

q1 [keyword trap] Can a goal be scored directly from a kick-off?
  gold chunk ['c0099']  | baseline sent ['c0167', 'c0103', 'c0170']  | rerank sent ['c0099', 'c0167', 'c0154']
  GOLD     : Yes, against the opponents. If the ball goes directly into the kicker's own goal, a corner kick is awarded to the opponents.
  ANSWER 1 : The context does not contain the answer.
  ANSWER 2 : Yes, a goal may be scored directly against the opponents from the kick-off. However, if the ball directly enters the kicker’s goal, a corner kick is awarded to the opponents.

q2 [exception] Is a goalkeeper sent off for handling the ball inside their own penalty area to stop a goal?
  gold chunk ['c0138']  | baseline sent ['c0155', 'c0134', 'c0107']  | rerank sent ['c0138', 'c0126', 'c0155']
  GOLD     : No. A deliberate handball that denies a goal or an obvious goal-scoring opportunity is a sending-off wherever it happens, except for a goalkeeper within their own penalty area.
  ANSWER 1 : The context does not 

## Stage 8 — Latency

Timed **one step at a time, never in parallel**, after a warm-up pass (the first call to a model is slower
because of lazy initialisation). Each retrieval step is timed over every query, 5 repeats, and the table
reports the **median**. Both encoders run on the CPU here.

The LLM is a network call to Google, so its time includes the internet and Google's load. It is the time
of the successful call only, without the throttle waits or retries, and it is **not comparable** with the
local steps. It was measured when the answers were first generated (it is stored in `answers.json`), so
if answers were reused from disk the figure is from that earlier run.

In [18]:
import statistics

REPEATS = 5
questions = [q["question"] for q in queries]

# warm-up: one throw-away pass so first-call overhead is not counted
for qs in questions[:2]:
    rerank(qs, retrieve(qs))

t_retrieve, t_rerank = [], []
for qs in questions:
    for _ in range(REPEATS):
        t0 = time.perf_counter(); cands = retrieve(qs); t1 = time.perf_counter()
        rerank(qs, cands); t2 = time.perf_counter()
        t_retrieve.append((t1 - t0) * 1000)
        t_rerank.append((t2 - t1) * 1000)

llm = {p: [a["llm_seconds"] for a in answers.values() if a["pipeline"] == p] for p in ("baseline", "rerank")}
med = statistics.median
ret_ms, rr_ms = med(t_retrieve), med(t_rerank)
latency = pd.DataFrame([
    {"step": "Bi-Encoder retrieval (embed question + search 230 chunks)", "baseline_ms": ret_ms, "rerank_ms": ret_ms},
    {"step": "Cross-Encoder re-ranking of the 10 candidates",            "baseline_ms": 0.0,    "rerank_ms": rr_ms},
    {"step": "Retrieval total (local, CPU)",                             "baseline_ms": ret_ms, "rerank_ms": ret_ms + rr_ms},
    {"step": "LLM call (Gemini API, median, network included)",          "baseline_ms": med(llm["baseline"]) * 1000,
                                                                          "rerank_ms": med(llm["rerank"]) * 1000},
]).round(1)
latency.to_csv(ROOT / "results" / "latency.csv", index=False)
print(f"median over {len(t_retrieve)} timings ({len(questions)} queries x {REPEATS} repeats)")
print(f"re-ranking costs {rr_ms / ret_ms:.1f}x the retrieval step on top of it")
latency

median over 30 timings (6 queries x 5 repeats)
re-ranking costs 15.1x the retrieval step on top of it


,step,baseline_ms,rerank_ms
0,Bi-Encoder retrieval (embed question + search ...,6.0,6.0
1,Cross-Encoder re-ranking of the 10 candidates,0.0,90.2
2,"Retrieval total (local, CPU)",6.0,96.2
3,"LLM call (Gemini API, median, network included)",3990.0,5565.0
